# Disease Prediction Model — Training on Google Colab

This notebook trains the Bio_ClinicalBERT disease classifier on a **free Colab GPU**.

**Before running:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload `DiseaseAndSymptoms.csv` when prompted

Expected training time: **~15 minutes** on T4 GPU (vs 20+ hours on CPU)

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers torch accelerate scikit-learn

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Step 2: Upload DiseaseAndSymptoms.csv
from google.colab import files
import os

if not os.path.exists('DiseaseAndSymptoms.csv'):
    print("Please upload DiseaseAndSymptoms.csv")
    uploaded = files.upload()
else:
    print("DiseaseAndSymptoms.csv already present")

In [ ]:
import csv
import json
import random
import re
from pathlib import Path

import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CSV_PATH = Path('DiseaseAndSymptoms.csv')
OUTPUT_DIR = Path('symptom_disease_model')
BASE_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'

In [ ]:
# Step 3: Load and prepare data

def clean_symptom(raw):
    s = raw.strip().lower()
    return re.sub(r'\s+', ' ', s)

def load_dataset(csv_path):
    rows = []
    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            disease = row[0].strip()
            if not disease:
                continue
            symptoms = [clean_symptom(c) for c in row[1:] if clean_symptom(c)]
            if symptoms:
                rows.append({'disease': disease, 'symptoms': symptoms})
    return rows

raw_rows = load_dataset(CSV_PATH)
print(f"Loaded {len(raw_rows)} rows")

diseases = sorted(set(r['disease'] for r in raw_rows))
label2id = {d: i for i, d in enumerate(diseases)}
id2label = {i: d for d, i in label2id.items()}
num_labels = len(diseases)
print(f"{num_labels} disease classes")

In [ ]:
# Step 4: Data augmentation

TEMPLATES = [
    lambda syms: ', '.join(syms),
    lambda syms: 'I have ' + ', '.join(s.replace('_', ' ') for s in syms),
    lambda syms: 'symptoms: ' + ', '.join(s.replace('_', ' ') for s in syms),
    lambda syms: 'experiencing ' + ' and '.join(s.replace('_', ' ') for s in syms),
    lambda syms: 'I am suffering from ' + ', '.join(s.replace('_', ' ') for s in syms),
    lambda syms: 'patient reports ' + ', '.join(s.replace('_', ' ') for s in syms),
    lambda syms: 'diagnosed with ' + ' and '.join(s.replace('_', ' ') for s in syms[:3])
    + (', ' + ', '.join(s.replace('_', ' ') for s in syms[3:]) if len(syms) > 3 else ''),
]

texts, labels = [], []
for row in raw_rows:
    lid = label2id[row['disease']]
    for tpl in TEMPLATES:
        texts.append(tpl(row['symptoms']))
        labels.append(lid)
    shuffled = row['symptoms'].copy()
    random.shuffle(shuffled)
    texts.append(', '.join(shuffled))
    labels.append(lid)

print(f"{len(texts):,} training examples after augmentation")

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)
print(f"Train: {len(train_texts):,}  Val: {len(val_texts):,}")

In [ ]:
# Step 5: Tokenize

class SymptomDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length', max_length=max_length
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print(f"Loading {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=num_labels, id2label=id2label, label2id=label2id
)

train_ds = SymptomDataset(train_texts, train_labels, tokenizer)
val_ds = SymptomDataset(val_texts, val_labels, tokenizer)
print("Tokenization complete")

In [ ]:
# Step 6: Train

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    return {'accuracy': float(acc)}

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=6,
    per_device_train_batch_size=32,      # Larger batch on GPU
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=25,
    seed=SEED,
    fp16=torch.cuda.is_available(),       # Mixed precision on GPU
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

In [ ]:
# Step 7: Evaluate
print("\n" + "="*60)
print("VALIDATION RESULTS")
print("="*60)

preds_output = trainer.predict(val_ds)
preds = np.argmax(preds_output.predictions, axis=-1)
target_names = [id2label[i] for i in range(num_labels)]
print(classification_report(val_labels, preds, target_names=target_names, zero_division=0))

In [ ]:
# Step 8: Save model
print(f"Saving model to {OUTPUT_DIR}...")
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

with open(OUTPUT_DIR / 'training_meta.json', 'w') as f:
    json.dump({
        'base_model': BASE_MODEL,
        'num_labels': num_labels,
        'label2id': label2id,
        'id2label': id2label,
        'epochs': 6,
        'augmented': True,
        'total_train_examples': len(train_texts),
        'total_val_examples': len(val_texts),
    }, f, indent=2)

print("Model saved!")

In [ ]:
# Step 9: Download the model
# Zip the model folder and download it to your local machine
import shutil

# Remove checkpoints to reduce zip size (we only need the final model)
checkpoints_dir = OUTPUT_DIR / 'checkpoints'
if checkpoints_dir.exists():
    shutil.rmtree(checkpoints_dir)
    print("Removed checkpoint folders to reduce size")

shutil.make_archive('symptom_disease_model', 'zip', '.', 'symptom_disease_model')
print(f"\nZip size: {os.path.getsize('symptom_disease_model.zip') / 1e6:.1f} MB")

files.download('symptom_disease_model.zip')
print("\nDownload started! After download:")
print("1. Unzip into backend/symptom_disease_model/")
print("2. Restart your backend server")
print("3. Test with: python predict_disease.py '{\"selectedSymptoms\":[\"itching\",\"skin_rash\"]}'")

---

## Quick Test (Optional)

Run a quick prediction to verify the model works before downloading.

In [ ]:
from transformers import pipeline as hf_pipeline

clf = hf_pipeline('text-classification', model=model, tokenizer=tokenizer,
                   return_all_scores=True, device=0 if torch.cuda.is_available() else -1)

test_cases = [
    'itching, skin_rash, nodal_skin_eruptions',
    'continuous_sneezing, shivering, chills, watering_from_eyes',
    'chest_pain, cough, vomiting, stomach_pain',
    'high_fever, headache, muscle_pain, joint_pain',
    'I have itching and skin rash with nodal skin eruptions',
]

print(f"{'Input':<60} {'Prediction':<35} {'Confidence'}")
print('-' * 110)
for text in test_cases:
    preds = clf(text)
    if isinstance(preds[0], list):
        preds = preds[0]
    best = max(preds, key=lambda x: x['score'])
    print(f"{text[:58]:<60} {best['label']:<35} {best['score']:.4f}")